# Mode3: 近 5 年 rodent LFP 在 2 选 1 / choice 任务中的分析流程综述

- 检索时间窗: 2021-04-04 到 2026-04-04
- 目标: 为 `Trial_Level_Mode3_GUI` 整理一份适合 rodent LFP + 认知神经科学 2 选 1 任务的分析流程、公开文档与代码入口。
- 说明: 直接公开且完整的“rodent + LFP + 2AFC + 全分析代码”资源仍然偏少，所以这里的推荐流程是基于近 5 年 rodent choice / discrimination LFP 论文，加上 IBL / AllenSDK / MNE / NeuroDSP 的官方文档和代码生态做的综合推断。


## 一句话结论

近 5 年 rodent LFP 在 choice / discrimination 任务中的主流流程已经比较稳定：

1. 连续记录与行为日志做硬件或时间戳同步。
2. 做丢包 / 饱和 / 运动伪迹检查，必要时插值，但不改变原始长度。
3. 做 LFP 提取、重参考、notch / band-pass。
4. 以 trial 和更细的 task epochs 为单位对齐，例如 baseline / stimulus / delay / choice / outcome / ITI。
5. 先做单通道功率特征，再做跨区或跨通道同步特征，例如 coherence、phase locking、PAC、CSD。
6. 统计上优先比较正确 vs 错误、左 vs 右、规则 A vs B、学习前 vs 学习后，并做 baseline normalization。
7. 若要做 2 选 1 解码，优先使用严格 trial-level cross-validation，避免同一 trial 的相邻时间窗泄漏到训练集和测试集。


## 近 5 年里最有参考价值的 rodent LFP 文献

| 来源 | 任务/动物 | 对 Mode3 最有用的点 |
| --- | --- | --- |
| [Ruikes et al., eNeuro, 2024](https://pubmed.ncbi.nlm.nih.gov/38621992/) | rat object discrimination task，多区同步记录 | 用 500 ms 级别 task epochs 分离 baseline、object discrimination、reward approach、reward outcome、ITI；重点看 theta coherence 和 spike-LFP entrainment，适合借鉴到你这里的 left/right choice 与 outcome 分层。 |
| [Ye et al., Oxford Open Neuroscience, 2023](https://pmc.ncbi.nlm.nih.gov/articles/PMC10348740/) | rat visual discrimination + reversal learning | 同时记录 ACC / OFC LFP，把 trial 明确切成多个行为阶段，再用 theta(5-10 Hz) 区分 correct / incorrect 与反转学习阶段；说明“trial 分段 + 准确率/反应时联合分析”是很强的范式。 |
| [Oberto et al., Frontiers in Cellular Neuroscience, 2023](https://www.frontiersin.org/journals/cellular-neuroscience/articles/10.3389/fncel.2023.1131313/full) | mouse operant olfactory / visual discrimination | 不只看单一 theta，直接把 slow、4 Hz、theta、alpha、low gamma、high gamma 拆开，并按 pre-cue / post-choice / outcome 等 task events 做 phase-locking 分析；很适合你后面扩展相位同步和频带特异分析。 |
| [de Almeida-Filho et al., Frontiers in Molecular Neuroscience, 2022](https://pmc.ncbi.nlm.nih.gov/articles/PMC9811406/) | rat object displacement discrimination / recognition memory | 方法写得很实：原始信号下采样到 1 kHz 得到 LFP，theta/gamma 带通，`spectrogram` 做时频，`pwelch` 做 PSD，再做 theta-gamma PAC，并且先按动物内归一化再比较条件。虽然不是经典 2AFC，但处理链条很值得复用。 |

### 从这些论文里抽出来的共识

- `trial-level` 对齐比单纯整段谱分析更重要，尤其是 choice / outcome 相关结论。
- theta 往往是 choice、delay、反馈最稳定的起点，但 4 Hz、beta、low gamma、high gamma 也常常携带额外信息。
- 近期论文越来越强调“功率 + 同步 + 相位”三层并行，而不是只做单个 band power。
- 对 rodent 行为任务，最关键的不是 fancy 模型，而是 task state 切得准、同步做得稳、试次标签干净。


## 官方文档和可直接复用的代码入口

| 资源 | 作用 |
| --- | --- |
| [IBL ONE Quick Start](https://docs.internationalbrainlab.org/notebooks_external/one_quickstart.html) | 用公开鼠类 2AFC 数据做 session 搜索和下载的起点。 |
| [IBL Loading Trials Data](https://docs.internationalbrainlab.org/notebooks_external/loading_trials_data.html) | 直接展示如何取 `choice`、`response_times`、`feedbackType`、`probabilityLeft` 等 2AFC 行为变量。 |
| [IBL Loading Ephys Data (AP and LFP band)](https://docs.internationalbrainlab.org/notebooks_external/loading_ephys_data.html) | 展示如何读取 LFP RMS / PSD，并按频段查看低频功率。 |
| [ONE GitHub](https://github.com/int-brain-lab/ONE) / [ibllib GitHub](https://github.com/int-brain-lab/ibllib) | 公开、持续维护的 rodent choice + ephys Python 代码生态。 |
| [AllenSDK Visual Behavior Neuropixels LFP Analysis](https://allensdk.readthedocs.io/en/stable/_static/examples/nb/visual_behavior_neuropixels_LFP_analysis.html) | 展示 LFP 读取、时间对齐、空间对齐、CSD，可直接借鉴 event-aligned LFP 组织方式。 |
| [Allen SWDB LFP Tutorial](https://allenswdb.github.io/physiology/ephys/visual-behavior/VBN-Tutorial-Analyzing_LFP_Data.html) | Jupyter notebook 风格教程，适合直接模仿 notebook 结构。 |
| [AllenSDK GitHub](https://github.com/AllenInstitute/AllenSDK) | Allen 公开数据读取与分析主仓库。 |
| [MNE Epochs overview](https://mne.tools/stable/auto_tutorials/epochs/10_epochs_overview.html) | 最适合把连续 LFP 改造成 `n_epochs x n_channels x n_times` 的标准结构。 |
| [MNE time-frequency simulated](https://mne.tools/stable/auto_examples/time_frequency/time_frequency_simulated.html) | 对比 Morlet / multitaper / Hilbert 的时频分析入口。 |
| [MNE sensor time-frequency tutorial](https://mne.tools/stable/auto_tutorials/time-freq/20_sensors_time_frequency.html) | 演示 power 与 ITC，非常适合 event-locked LFP。 |
| [mne-python GitHub](https://github.com/mne-tools/mne-python) | 如果你要继续往 epoch metadata、统计检验、解码走，MNE 是最顺手的 Python 框架之一。 |
| [NeuroDSP Spectral Power tutorial](https://neurodsp-tools.github.io/neurodsp/auto_tutorials/spectral/plot_SpectralPower.html) | 官方示例直接用 rat hippocampal LFP 演示 PSD。 |
| [NeuroDSP Spectral Variance tutorial](https://neurodsp-tools.github.io/neurodsp/auto_tutorials/spectral/plot_SpectralVariance.html) | 用 SCV 看频谱稳定性，适合做数据质量检查。 |
| [NeuroDSP IRASA tutorial](https://neurodsp-tools.github.io/neurodsp/auto_tutorials/aperiodic/plot_IRASA.html) | 后续如果你想把 oscillatory peak 和 1/f 成分拆开，这个很好用。 |
| [neurodsp GitHub](https://github.com/neurodsp-tools/neurodsp) | 轻量、适合 LFP 的频谱/节律分析工具箱。 |

### 我对这些资源的判断

- 如果你要找“最接近 rodent 2AFC 的公开任务代码”，优先看 IBL。
- 如果你要找“最接近 notebook 教程、容易照着改的 LFP 示例”，优先看 AllenSDK / Allen SWDB。
- 如果你要把自己的 EDF + behavior 数据做成标准 trial-based pipeline，MNE 最适合做 epochs 和时频。
- 如果你要快速做谱、burst、aperiodic、频带质量检查，NeuroDSP 最轻便。


## 对你当前 Mode3 代码的映射

你现在仓库里已经具备的部分：

- EDF 读取与 `Trial.txt` / `Tevent.txt` 解析。
- 用阈值 `-1000` 检查缺失并做线性插值。
- 对 raw 通道做 4.5-5.5 kHz 带通，再检测同步脉冲。
- 用同步脉冲把 neural data 与 behavior trial 对齐。
- GUI 里已经有 spectrogram、theta/gamma 能量、lick event、sensor 对齐展示。

还建议在 notebook / 后续脚本里补强的部分：

1. 把每个 trial 的元数据整理成一张 DataFrame，至少包含 `trial_type`、`trial_outcome`、`rule`、`is_earlylick`、`start_ts`、`end_ts`。
2. 在 trial 内继续按状态机切分成更细的 task epochs，而不是只保留整段。
3. 增加重参考选项，例如 common median reference 或 per-shank median。
4. 增加 notch / line noise 检查和运动伪迹排除规则。
5. 除了 theta/gamma 平均能量，再补 `beta`、`4 Hz`、`coherence`、`ITC`、`PAC`。
6. 解码时用 trial-level split，而不是随机打散时间片，避免信息泄漏。

下面的代码单元先给你一个 starter workflow，用你仓库的 `ExampleData` 直接接上现有 `DataProcessor`。


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

from data_processing import DataProcessor, parse_trial_file

BASE = Path.cwd()
NEURAL_DIR = BASE / "ExampleData" / "NeuralFile"
BEHAV_DIR = BASE / "ExampleData" / "BehaviorFile"

BASE, NEURAL_DIR.exists(), BEHAV_DIR.exists()


In [ ]:
dp = DataProcessor()
extracted_trials, stats, missing_log = dp.batch_process_directories(
    str(NEURAL_DIR),
    str(BEHAV_DIR),
)
trial_meta = parse_trial_file(BEHAV_DIR / "Trial.txt")

summary_rows = []
for key, trial_data in sorted(extracted_trials.items()):
    trial_id = int(re.search(r"Trial_(\d+)$", key).group(1))
    meta = trial_meta.get(trial_id)
    lfp_t_ms, lfp_v = trial_data["lfp"]
    summary_rows.append(
        {
            "trial_key": key,
            "trial_id": trial_id,
            "trial_type": getattr(meta, "trial_type", np.nan),
            "trial_outcome": getattr(meta, "trial_outcome", np.nan),
            "rule": getattr(meta, "rule", np.nan),
            "is_earlylick": getattr(meta, "is_earlylick", np.nan),
            "lfp_channels": lfp_v.shape[0],
            "lfp_samples": lfp_v.shape[1],
            "lfp_t_min_ms": float(lfp_t_ms[0]),
            "lfp_t_max_ms": float(lfp_t_ms[-1]),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.head(), stats, missing_log[:5]


In [ ]:
def infer_fs_from_time_ms(time_ms):
    dt_ms = np.median(np.diff(time_ms))
    return 1000.0 / dt_ms

example_key = sorted(extracted_trials)[0]
lfp_t_ms, lfp_v = extracted_trials[example_key]["lfp"]
fs_lfp = infer_fs_from_time_ms(lfp_t_ms)
channel = 0

f, t, sxx = signal.spectrogram(
    lfp_v[channel],
    fs=fs_lfp,
    window="hann",
    nperseg=int(fs_lfp * 0.5),
    noverlap=int(fs_lfp * 0.25),
    scaling="density",
    mode="psd",
)

freq_mask = f <= 150

plt.figure(figsize=(10, 4))
plt.pcolormesh(
    t * 1000 + lfp_t_ms[0],
    f[freq_mask],
    10 * np.log10(sxx[freq_mask] + 1e-12),
    shading="auto",
)
plt.axvline(0, color="white", linestyle="--", linewidth=1)
plt.xlabel("Time from trial start (ms)")
plt.ylabel("Frequency (Hz)")
plt.title(f"{example_key} | ch{channel} spectrogram")
plt.colorbar(label="Power (dB)")
plt.tight_layout()


In [ ]:
TASK_WINDOWS_MS = {
    "baseline": (-1000, 0),
    "sample_or_stim": (0, 500),
    "delay": (500, 1500),
    "response": (1500, 2500),
    "outcome": (2500, 3500),
}

BANDS = {
    "theta": (4, 8),
    "beta": (13, 30),
    "low_gamma": (30, 55),
    "high_gamma": (65, 100),
}

def bandpower_welch(sig, fs, band):
    nperseg = min(len(sig), max(128, int(fs)))
    freqs, psd = signal.welch(sig, fs=fs, nperseg=nperseg)
    mask = (freqs >= band[0]) & (freqs <= band[1])
    if not np.any(mask):
        return np.nan
    return float(np.trapz(psd[mask], freqs[mask]))

rows = []
for key, trial_data in sorted(extracted_trials.items()):
    trial_id = int(re.search(r"Trial_(\d+)$", key).group(1))
    meta = trial_meta.get(trial_id)
    lfp_t_ms, lfp_v = trial_data["lfp"]
    fs = infer_fs_from_time_ms(lfp_t_ms)

    for phase, (start_ms, end_ms) in TASK_WINDOWS_MS.items():
        time_mask = (lfp_t_ms >= start_ms) & (lfp_t_ms < end_ms)
        if time_mask.sum() < max(32, int(fs * 0.25)):
            continue

        phase_sig = lfp_v[:, time_mask]
        mean_sig = phase_sig.mean(axis=0)

        row = {
            "trial_key": key,
            "trial_id": trial_id,
            "trial_type": getattr(meta, "trial_type", np.nan),
            "trial_outcome": getattr(meta, "trial_outcome", np.nan),
            "rule": getattr(meta, "rule", np.nan),
            "phase": phase,
        }
        for band_name, band in BANDS.items():
            row[f"{band_name}_power"] = bandpower_welch(mean_sig, fs, band)
        rows.append(row)

bandpower_df = pd.DataFrame(rows)
bandpower_df.head()


## 后续建议

1. 把 `TASK_WINDOWS_MS` 从固定窗口改成基于 `states_history` 的真实状态机边界。
2. 对每个 phase 做 baseline normalization，例如 `(power - baseline) / baseline` 或 dB change。
3. 新增 left vs right、correct vs error、rule A vs B 的分组统计。
4. 如果要做网络层指标，下一步优先加 `scipy.signal.coherence` 或 MNE 的 ITC / multitaper TFR。
5. 如果要做 choice decoding，建议把 `bandpower_df` 或单 trial TFR 压成 trial-level features，再做 stratified cross-validation。

这份 notebook 的定位是“综述 + starter code”。后面如果你愿意，我们可以直接继续把它扩成真正跑你 Mode3 数据的正式分析 notebook。
